In [3]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

# This is required to run multiple processes with JAX.
from multiprocessing import set_start_method
set_start_method('forkserver', force=True)

In [38]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from tqdm import tqdm
from pathlib import Path

from config import Config, AssimConfig
import data

In [41]:
from train import Trainer

run_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/runs/sf_grnn/era5_20251206_231851")
trainer = Trainer.load_from_run_or_state(run_dir)


assim_path = Path("/nas/cee-water/cjgleason/ted/swot-ml/runs/_assim/swot_large.yml")
assim_cfg = AssimConfig.from_file(assim_path)
trainer.cfg.add_assimilation(assim_cfg)
cfg = trainer.cfg

manager = data.DynamicCacheManager(cfg)
cache_dir = manager.create_cache('train')
dataset = data.CachedBasinGraphDataset(cfg, cache_dir, "train")
trainer.training_dl = data.CachedBasinGraphDataLoader(cfg, dataset)

# Update the model with the new assimilation modules
new_model = trainer.model
for new_group in assim_cfg.new_features.keys():
    n_group_features = len(dataset.features.dynamic[new_group])
    group_network_sizes = assim_cfg.model_args.get(new_group, {})
    new_model.add_assimilator(new_group, n_group_features, group_network_sizes)
    trainer.cfg.model_args.sparse_sizes[new_group] = n_group_features
trainer.replace_model(new_model)

Loading model from /nas/cee-water/cjgleason/ted/swot-ml/runs/sf_grnn/era5_20251206_231851/checkpoints/step_050000
Model contains 379,361 parameters
Loading trainer from checkpoint step_050000
Logging at /nas/cee-water/cjgleason/ted/swot-ml/runs/sf_grnn/era5_20251206_231851
Caches will be stored at: /scratch4/workspace/tlanghorst_umass_edu-swot-ml-zarr/zbs_batched/_cache4/70d57ba28642623e
Calculating training statistics for encoding and normalization...
✅ All caches created and indexed.
Calculating training statistics for encoding and normalization...
Loading static attributes...Done!
Loading basin graphs...Done!
Loading optimized indices for train...Done!
Dataloader using 12 parallel CPU worker(s).


/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [43]:
trainer.opt_state[0][1].count

Array(0, dtype=int32)

In [52]:
trainer.freeze_components(["dense_embedders", "fusion_norm", "spat_temp_lstm"])

trainer.filter_spec.dense_embedders

{'era5': MLP(
   layers=(
     Linear(
       weight=False, bias=False, in_features=10, out_features=64, use_bias=True
     ),
     Linear(
       weight=False, bias=False, in_features=64, out_features=64, use_bias=True
     )
   ),
   activation=False,
   final_activation=False,
   use_bias=True,
   use_final_bias=True,
   in_size=10,
   out_size=64,
   width_size=64,
   depth=1
 )}